# Spine Skeleton vs Dendrite Skeleton

Загружает mesh шипика, запаивает дырки, строит CGAL skeleton шипика, подгружает `branch_skeleton.npy` дендрита, визуализирует оба skeleton на одном 3D-графике и считает профиль кратчайших расстояний от 10 равномерных точек skeleton шипика до skeleton дендрита.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path, PureWindowsPath
from typing import Iterable

import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import trimesh


def find_repo_root(start: Path | None = None) -> Path:
    """Find repository root from notebook cwd."""
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src" / "dendrite_analysis").exists() and (candidate / "CGAL").exists():
            return candidate
    return Path.cwd().resolve().parents[1]


REPO_ROOT = find_repo_root()
for path in (REPO_ROOT, REPO_ROOT / "src"):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from CGAL.CGAL_Polygon_mesh_processing import Polylines
from CGAL.CGAL_Surface_mesh_skeletonization import surface_mesh_skeletonization
from src.dendrite_analysis.mesh_repair import repair_mesh
from src.dendrite_analysis.network import _segments_from_skeleton_object
from src.spine_analysis.mesh.utils import v_f_to_mesh_isolated


def resolve_data_path(raw_path: str) -> Path:
    """Resolve a Windows dataset path, optionally remapped on non-Windows systems.

    If this notebook is run on macOS/Linux, set MINNIE65_DRIVE_ROOT to the
    mounted root corresponding to O:\\, for example /Volumes/O.
    """
    direct = Path(raw_path)
    if direct.exists() or os.name == "nt":
        return direct

    drive_root = os.environ.get("MINNIE65_DRIVE_ROOT")
    if drive_root:
        win_path = PureWindowsPath(raw_path)
        return Path(drive_root, *win_path.parts[1:])
    return direct


SPINE_MESH_PATH = resolve_data_path(
    r"O:\Datasets\Minnie65\results\Combined_Data\864691134886335738\limb_001\branch_002\spines\spine_000.off"
)
DENDRITE_SKELETON_PATH = resolve_data_path(
    r"O:\Datasets\Minnie65\results\Combined_Data\864691134886335738\limb_001\branch_002\branch_skeleton.npy"
)

SPINE_MESH_PATH, DENDRITE_SKELETON_PATH


In [ ]:
def ensure_file(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"{label} not found: {path}\n"
            "If you are running outside Windows, set MINNIE65_DRIVE_ROOT to the mounted O: drive root."
        )


def load_and_repair_spine_mesh(mesh_path: Path) -> trimesh.Trimesh:
    """Load spine mesh with trimesh and repair holes/winding."""
    ensure_file(mesh_path, "Spine mesh")
    mesh = trimesh.load_mesh(str(mesh_path), process=False)
    if not isinstance(mesh, trimesh.Trimesh):
        raise TypeError(f"Expected trimesh.Trimesh, got {type(mesh).__name__}")

    repaired, report_before, report_after = repair_mesh(
        mesh,
        fix_normals=True,
        fill_holes=True,
        remove_degenerate=True,
        stitch_borders=True,
        verbose=True,
    )
    if not isinstance(repaired, trimesh.Trimesh):
        repaired = trimesh.Trimesh(
            vertices=np.asarray(repaired.vertices, dtype=float),
            faces=np.asarray(repaired.faces, dtype=int),
            process=False,
        )
    return repaired


def trimesh_to_cgal_polyhedron(mesh: trimesh.Trimesh):
    vertices = np.asarray(mesh.vertices, dtype=float)
    faces = np.asarray(mesh.faces, dtype=int)
    return v_f_to_mesh_isolated(vertices, faces)


def cgal_skeleton_segments(polyhedron) -> list[tuple[np.ndarray, np.ndarray]]:
    """Build CGAL mean-curvature-flow skeleton and return straight segments."""
    if not bool(polyhedron.is_closed()):
        raise RuntimeError("CGAL skeletonization requires a closed/watertight mesh after repair.")

    skeleton_polylines = Polylines()
    correspondence_polylines = Polylines()
    surface_mesh_skeletonization(polyhedron, skeleton_polylines, correspondence_polylines)

    segments: list[tuple[np.ndarray, np.ndarray]] = []
    for polyline in skeleton_polylines:
        points = [np.array([p.x(), p.y(), p.z()], dtype=float) for p in polyline]
        for start, end in zip(points[:-1], points[1:]):
            if np.linalg.norm(end - start) > 0:
                segments.append((start, end))
    if not segments:
        raise RuntimeError("CGAL returned an empty spine skeleton.")
    return segments


def load_dendrite_skeleton_segments(path: Path) -> list[tuple[np.ndarray, np.ndarray]]:
    ensure_file(path, "Dendrite skeleton")
    skeleton = np.load(path, allow_pickle=True)
    segments = _segments_from_skeleton_object(skeleton)
    if not segments:
        raise RuntimeError(f"No valid 3-D segments found in {path}")
    return [(np.asarray(a, dtype=float), np.asarray(b, dtype=float)) for a, b in segments]


In [ ]:
def graph_from_segments(segments: Iterable[tuple[np.ndarray, np.ndarray]], ndigits: int = 6) -> nx.Graph:
    graph = nx.Graph()
    key_to_node: dict[tuple[float, float, float], int] = {}

    def node_for(point: np.ndarray) -> int:
        key = tuple(np.round(np.asarray(point, dtype=float), ndigits).tolist())
        if key not in key_to_node:
            node_id = len(key_to_node)
            key_to_node[key] = node_id
            graph.add_node(node_id, pos=np.asarray(point, dtype=float))
        return key_to_node[key]

    for start, end in segments:
        u = node_for(start)
        v = node_for(end)
        if u == v:
            continue
        length = float(np.linalg.norm(end - start))
        if length <= 0:
            continue
        if graph.has_edge(u, v):
            graph[u][v]["length"] = min(graph[u][v]["length"], length)
        else:
            graph.add_edge(u, v, length=length)
    return graph


def longest_path_polyline(segments: list[tuple[np.ndarray, np.ndarray]]) -> np.ndarray:
    """Return the longest weighted terminal-to-terminal path as a polyline."""
    graph = graph_from_segments(segments)
    if graph.number_of_nodes() < 2:
        raise RuntimeError("Skeleton graph has fewer than two nodes.")

    best_length = -np.inf
    best_path: list[int] | None = None
    for component_nodes in nx.connected_components(graph):
        sub = graph.subgraph(component_nodes)
        terminals = [node for node in sub.nodes if sub.degree(node) == 1]
        candidates = terminals if len(terminals) >= 2 else list(sub.nodes)
        for i, source in enumerate(candidates):
            lengths, paths = nx.single_source_dijkstra(sub, source, weight="length")
            for target in candidates[i + 1:]:
                distance = lengths.get(target, -np.inf)
                if distance > best_length:
                    best_length = distance
                    best_path = paths[target]

    if best_path is None:
        raise RuntimeError("Cannot identify a longest path in the spine skeleton.")
    return np.vstack([graph.nodes[node]["pos"] for node in best_path])


def sample_polyline_by_arclength(polyline: np.ndarray, n_points: int = 10) -> np.ndarray:
    """Sample a polyline uniformly by arclength, including both endpoints."""
    polyline = np.asarray(polyline, dtype=float)
    if len(polyline) < 2:
        raise ValueError("Polyline must contain at least two points.")

    segment_lengths = np.linalg.norm(np.diff(polyline, axis=0), axis=1)
    keep = np.r_[True, segment_lengths > 0]
    polyline = polyline[keep]
    segment_lengths = np.linalg.norm(np.diff(polyline, axis=0), axis=1)
    cumulative = np.r_[0.0, np.cumsum(segment_lengths)]
    total = float(cumulative[-1])
    if total == 0:
        raise ValueError("Polyline length is zero.")

    targets = np.linspace(0.0, total, n_points)
    samples = []
    for target in targets:
        seg_idx = min(np.searchsorted(cumulative, target, side="right") - 1, len(segment_lengths) - 1)
        local = 0.0 if segment_lengths[seg_idx] == 0 else (target - cumulative[seg_idx]) / segment_lengths[seg_idx]
        samples.append(polyline[seg_idx] * (1.0 - local) + polyline[seg_idx + 1] * local)
    return np.vstack(samples)


def point_to_segment_distance(point: np.ndarray, start: np.ndarray, end: np.ndarray) -> float:
    point = np.asarray(point, dtype=float)
    start = np.asarray(start, dtype=float)
    end = np.asarray(end, dtype=float)
    vector = end - start
    denom = float(np.dot(vector, vector))
    if denom == 0:
        return float(np.linalg.norm(point - start))
    t = float(np.clip(np.dot(point - start, vector) / denom, 0.0, 1.0))
    projection = start + t * vector
    return float(np.linalg.norm(point - projection))


def distance_to_segments(point: np.ndarray, segments: list[tuple[np.ndarray, np.ndarray]]) -> float:
    return min(point_to_segment_distance(point, start, end) for start, end in segments)


def orient_spine_path_from_dendrite(spine_path: np.ndarray, dendrite_segments: list[tuple[np.ndarray, np.ndarray]]) -> np.ndarray:
    """Orient spine path so point 0 is closest to the dendrite skeleton."""
    first_distance = distance_to_segments(spine_path[0], dendrite_segments)
    last_distance = distance_to_segments(spine_path[-1], dendrite_segments)
    if last_distance < first_distance:
        return spine_path[::-1].copy()
    return spine_path


In [ ]:
spine_mesh = load_and_repair_spine_mesh(SPINE_MESH_PATH)
print(f"Repaired spine mesh: vertices={len(spine_mesh.vertices)}, faces={len(spine_mesh.faces)}, watertight={spine_mesh.is_watertight}")

spine_polyhedron = trimesh_to_cgal_polyhedron(spine_mesh)
print(f"CGAL spine mesh closed: {bool(spine_polyhedron.is_closed())}")

spine_segments = cgal_skeleton_segments(spine_polyhedron)
dendrite_segments = load_dendrite_skeleton_segments(DENDRITE_SKELETON_PATH)

print(f"Spine skeleton segments: {len(spine_segments)}")
print(f"Dendrite skeleton segments: {len(dendrite_segments)}")


In [ ]:
spine_path = longest_path_polyline(spine_segments)
spine_path = orient_spine_path_from_dendrite(spine_path, dendrite_segments)
spine_sample_points = sample_polyline_by_arclength(spine_path, n_points=10)

profile = pd.DataFrame(
    {
        "point_index": np.arange(len(spine_sample_points), dtype=int),
        "x": spine_sample_points[:, 0],
        "y": spine_sample_points[:, 1],
        "z": spine_sample_points[:, 2],
        "distance_to_dendrite_skeleton": [
            distance_to_segments(point, dendrite_segments)
            for point in spine_sample_points
        ],
    }
)
profile


In [ ]:
def segment_trace(
    segments: list[tuple[np.ndarray, np.ndarray]],
    name: str,
    color: str,
    width: int = 5,
    dashed: bool = False,
    dash_count_per_segment: int = 6,
) -> go.Scatter3d:
    xs, ys, zs = [], [], []
    for start, end in segments:
        start = np.asarray(start, dtype=float)
        end = np.asarray(end, dtype=float)
        if dashed:
            # Scatter3d has limited native dash support, so draw only every other subsegment.
            for i in range(dash_count_per_segment):
                if i % 2:
                    continue
                a = start + (end - start) * (i / dash_count_per_segment)
                b = start + (end - start) * ((i + 1) / dash_count_per_segment)
                xs.extend([a[0], b[0], None])
                ys.extend([a[1], b[1], None])
                zs.extend([a[2], b[2], None])
        else:
            xs.extend([start[0], end[0], None])
            ys.extend([start[1], end[1], None])
            zs.extend([start[2], end[2], None])
    return go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,
        mode="lines",
        name=name,
        line=dict(color=color, width=width),
    )


fig = go.Figure()
fig.add_trace(segment_trace(dendrite_segments, "dendrite skeleton", "#2ca02c", width=5))
fig.add_trace(segment_trace(spine_segments, "spine skeleton", "#ff69b4", width=7, dashed=True))
fig.add_trace(
    go.Scatter3d(
        x=spine_sample_points[:, 0],
        y=spine_sample_points[:, 1],
        z=spine_sample_points[:, 2],
        mode="markers+text",
        name="10 sampled spine points",
        marker=dict(size=5, color="#ff69b4", symbol="circle"),
        text=[str(i) for i in range(len(spine_sample_points))],
        textposition="top center",
    )
)

all_points = np.vstack(
    [
        np.asarray([point for segment in dendrite_segments for point in segment], dtype=float),
        np.asarray([point for segment in spine_segments for point in segment], dtype=float),
        spine_sample_points,
    ]
)
center = all_points.mean(axis=0)
span = float(np.max(np.ptp(all_points, axis=0)))
if not np.isfinite(span) or span == 0:
    span = 1.0

fig.update_layout(
    title="Spine skeleton vs dendrite skeleton",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="data",
        xaxis=dict(range=[center[0] - span / 2, center[0] + span / 2]),
        yaxis=dict(range=[center[1] - span / 2, center[1] + span / 2]),
        zaxis=dict(range=[center[2] - span / 2, center[2] + span / 2]),
    ),
    legend=dict(x=0.02, y=0.98),
    width=950,
    height=760,
)
fig.show()


In [ ]:
print("Pairwise distance profile: sampled spine skeleton point -> nearest dendrite skeleton")
for row in profile.itertuples(index=False):
    print(
        f"point {row.point_index:02d}: "
        f"({row.x:.3f}, {row.y:.3f}, {row.z:.3f}) -> "
        f"{row.distance_to_dendrite_skeleton:.6f}"
    )
